In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/uni/pcam'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

Login OK


In [3]:
import timm
from huggingface_hub import hf_hub_download

# UNI özel yükleme - timm ile
model = timm.create_model(
    "hf-hub:MahmoodLab/UNI",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=True
)
model = model.cuda()
model.eval()
print("UNI loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

UNI loaded!
GPU memory: 1.21 GB


In [4]:
from torchvision import transforms

# UNI's self preprocessing not Imagenet's!
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])
print("UNI transform ready")

UNI transform ready


In [5]:
from torchvision.datasets import PCAM
from torch.utils.data import DataLoader

PCAM_DIR = '/content/drive/MyDrive/PRISM/datasets/pcam'

train_dataset = PCAM(root=PCAM_DIR, split='train', download=False, transform=transform)
val_dataset   = PCAM(root=PCAM_DIR, split='val',   download=False, transform=transform)
test_dataset  = PCAM(root=PCAM_DIR, split='test',  download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

Train: 262,144
Val:   32,768
Test:  32,768


In [6]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            features = model(images)  # UNI direkt forward
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 512/512 [1:02:13<00:00,  7.29s/it]


Train: (262144, 1024)
Extracting val features...


Extracting: 100%|██████████| 64/64 [04:53<00:00,  4.59s/it]


Val: (32768, 1024)
Extracting test features...


Extracting: 100%|██████████| 64/64 [05:46<00:00,  5.42s/it]

Test: (32768, 1024)


In [7]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy', train_labels)
np.save(f'{EMBED_DIR}/val_features.npy', val_features)
np.save(f'{EMBED_DIR}/val_labels.npy', val_labels)
np.save(f'{EMBED_DIR}/test_features.npy', test_features)
np.save(f'{EMBED_DIR}/test_labels.npy', test_labels)
print(f"Saved to: {EMBED_DIR}")

Saved to: /content/drive/MyDrive/PRISM/embeddings/uni/pcam


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece / len(y_true)

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])

    y_prob = clf.predict_proba(test_f)[:, 1]
    y_pred = clf.predict(test_f)

    return {
        'model': 'UNI', 'dataset': 'PCam',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob),
        'f1':    f1_score(test_l, y_pred),
        'brier': brier_score_loss(test_l, y_prob),
        'ece':   compute_ece(test_l, y_prob),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.9810 | ECE: 0.0690 | Brier: 0.0544
  1% | seed 123 | AUROC: 0.9806 | ECE: 0.0771 | Brier: 0.0582
  1% | seed 456 | AUROC: 0.9827 | ECE: 0.0758 | Brier: 0.0555
  5% | seed 42 | AUROC: 0.9825 | ECE: 0.0389 | Brier: 0.0486
  5% | seed 123 | AUROC: 0.9829 | ECE: 0.0441 | Brier: 0.0504
  5% | seed 456 | AUROC: 0.9847 | ECE: 0.0425 | Brier: 0.0471
  10% | seed 42 | AUROC: 0.9811 | ECE: 0.0377 | Brier: 0.0490
  10% | seed 123 | AUROC: 0.9836 | ECE: 0.0373 | Brier: 0.0477
  10% | seed 456 | AUROC: 0.9845 | ECE: 0.0364 | Brier: 0.0464
  25% | seed 42 | AUROC: 0.9800 | ECE: 0.0417 | Brier: 0.0504
  25% | seed 123 | AUROC: 0.9832 | ECE: 0.0373 | Brier: 0.0470
  25% | seed 456 | AUROC: 0.9824 | ECE: 0.0399 | Brier: 0.0484
  50% | seed 42 | AUROC: 0.9811 | ECE: 0.0410 | Brier: 0.0495
  50% | seed 123 | AUROC: 0.9812 | ECE: 0.0425 | Brier: 0.0500
  50% | seed 456 | AUROC: 0.9808 | ECE: 0.0444 | Brier: 0.0506
  100% | seed 42 | AUROC: 0.9786 | ECE: 0.0451 | Brier: 0.0525
  1

In [9]:
from scipy.optimize import minimize_scalar

def find_temperature(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])

    val_logits  = clf.decision_function(val_f).reshape(-1,1)
    val_logits  = np.hstack([-val_logits, val_logits])
    T           = find_temperature(val_logits, val_l)

    test_prob_raw = clf.predict_proba(test_f)[:, 1]
    ece_raw       = compute_ece(test_l, test_prob_raw)

    test_logits = clf.decision_function(test_f).reshape(-1,1)
    test_logits = np.hstack([-test_logits, test_logits])
    scaled      = test_logits / T
    exp_s       = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s = (exp_s / exp_s.sum(axis=1, keepdims=True))[:, 1]
    ece_scaled  = compute_ece(test_l, test_prob_s)

    return {
        'model': 'UNI', 'dataset': 'PCam',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc': roc_auc_score(test_l, test_prob_raw),
        'brier_raw': brier_score_loss(test_l, test_prob_raw),
        'brier_scaled': brier_score_loss(test_l, test_prob_s),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           1.3882   0.0740      0.0409           0.0331
0.05           1.9204   0.0418      0.0399           0.0019
0.10           2.1255   0.0372      0.0377          -0.0006
0.25           2.3568   0.0396      0.0369           0.0028
0.50           2.5613   0.0426      0.0389           0.0038
1.00           2.6635   0.0451      0.0410           0.0041


In [10]:
df.to_csv(f'{RESULTS_DIR}/uni_pcam_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/uni_pcam_temperature_scaling.csv', index=False)
print("Saved!")

Saved!
